# NanoGPT — ROCStories Narrative Experiment

**Course:** COMP4680/8650 — Advanced Topics in Machine Learning  
**Research Question:** Does narrative direction and structure matter to an autoregressive language model?

## Three Models, One Architecture

| Model | Data Format | Research Question |
|-------|------------|------------------|
| **M1 — Forward** | S1 → S2 → S3 → S4 → S5 | Baseline: standard narrative order |
| **M2 — Reverse** | S5 → S4 → S3 → S2 → S1 | H1: is temporal direction symmetric? |
| **M3 — Structured** | `<S1>` S1 `<S2>` S2 ... `<S5>` S5 | H2: do position markers help coherence? |

## How to Use This Notebook

**Before full training**, run the **Smoke Test** (Step 4) first — it runs 50 iterations per model in ~2 minutes total, verifying the entire pipeline works.

**Then run full training** (Step 5) — ~30-60 min per model on a free Colab T4 GPU.

**Runtime:** Make sure you have a GPU enabled: `Runtime → Change runtime type → GPU (T4 or better)`

---
⚠️ **Important:** Enable GPU before running! Check: `Runtime → Change runtime type → T4 GPU`

## Step 0: Setup — Install Dependencies & Mount Drive

Run this cell first. It installs required packages and mounts Google Drive to save checkpoints persistently.

In [ ]:
# ── 0a. Check GPU ──────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU detected:', result.stdout.strip())
else:
    print('WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

# ── 0b. Install packages ───────────────────────────────────────────────────
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q tiktoken transformers datasets pandas numpy tqdm wandb matplotlib

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
    print(f'bfloat16 supported: {torch.cuda.is_bf16_supported()}')

In [ ]:
# ── 0c. Mount Google Drive (saves checkpoints even if session crashes) ────
from google.colab import drive
drive.mount('/content/drive')

import os

# ── 0d. Create project directory on Drive ─────────────────────────────────
DRIVE_PROJECT = '/content/drive/MyDrive/nanoGPT_rocstories'
os.makedirs(DRIVE_PROJECT, exist_ok=True)
print(f'Project directory: {DRIVE_PROJECT}')

## Step 1: Get Project Files

**Option A** (recommended): Clone from GitHub after you push your repo.  
**Option B**: Upload a zip of the demo folder manually.

Choose one option below and run it.

In [ ]:
# ── Option A: Clone from GitHub ────────────────────────────────────────────
# Replace with your actual GitHub repo URL after pushing
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'

import os
DEMO_DIR = '/content/demo'

if not os.path.exists(DEMO_DIR):
    !git clone {GITHUB_REPO} /content/repo
    !cp -r /content/repo/demo /content/demo
    print('Cloned and copied demo folder.')
else:
    print('demo/ already exists, skipping clone.')

os.chdir(DEMO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls -la

In [ ]:
# ── Option B: Upload zip manually ─────────────────────────────────────────
# Uncomment and run this cell if you prefer to upload a zip

# from google.colab import files
# uploaded = files.upload()  # Upload demo.zip
# !unzip -q demo.zip -d /content/
# import os
# DEMO_DIR = '/content/demo'
# os.chdir(DEMO_DIR)
# print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── Verify project structure ──────────────────────────────────────────────
import os
DEMO_DIR = '/content/demo'
os.chdir(DEMO_DIR)

required_files = [
    'model.py', 'train.py', 'eval.py', 'sample.py', 'configurator.py',
    'data/rocstories/prepare.py',
    'config/train_m1_forward.py',
    'config/train_m2_reverse.py',
    'config/train_m3_structured.py',
]
all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    status = '✓' if exists else '✗ MISSING'
    print(f'  {status}  {f}')
    if not exists:
        all_ok = False

print()
print('All files present!' if all_ok else 'ERROR: Some files are missing. Check Step 1.')

## Step 2: Symlink Output Directories to Google Drive

This ensures checkpoints are saved to Drive and survive session restarts.

In [ ]:
import os

DEMO_DIR = '/content/demo'
os.chdir(DEMO_DIR)

# Create output dirs on Drive and symlink to demo/
for model_name in ['out-m1-forward', 'out-m2-reverse', 'out-m3-structured',
                   'out-test-m1', 'out-test-m2', 'out-test-m3']:
    drive_path = f'/content/drive/MyDrive/nanoGPT_rocstories/{model_name}'
    local_link = os.path.join(DEMO_DIR, model_name)

    os.makedirs(drive_path, exist_ok=True)

    # Create symlink only if it doesn't exist yet
    if not os.path.exists(local_link):
        os.symlink(drive_path, local_link)
        print(f'Linked: {model_name} → Drive')
    else:
        print(f'Exists: {model_name}')

print('\nDrive output directories ready.')

## Step 3: Download ROCStories Dataset

Download from HuggingFace and prepare all 3 data variants.

In [ ]:
# ── 3a. Download ROCStories CSV from HuggingFace ──────────────────────────
import os
os.chdir('/content/demo')

csv_path = 'data/rocstories/rocstories_train.csv'

if os.path.exists(csv_path):
    import pandas as pd
    df = pd.read_csv(csv_path)
    print(f'CSV already exists: {len(df):,} stories')
else:
    print('Downloading ROCStories from HuggingFace...')
    from datasets import load_dataset
    ds = load_dataset('mintujupally/ROCStories', split='train')
    df = ds.to_pandas()
    print(f'Downloaded {len(df):,} stories')
    print(f'Columns: {list(df.columns)}')

    # Save to expected location
    os.makedirs('data/rocstories', exist_ok=True)
    df.to_csv(csv_path, index=False)
    print(f'Saved to {csv_path}')

# Preview a sample story
import pandas as pd
df = pd.read_csv(csv_path)
print(f'\nDataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

sample = df.iloc[0]
print('\nSample story:')
for i in range(1, 6):
    print(f'  S{i}: {sample[f"sentence{i"}}')

In [ ]:
# ── 3b. Prepare all 3 data variants ──────────────────────────────────────
import os
os.chdir('/content/demo')

print('Preparing all 3 variants (forward, reverse, structured)...')
print('This takes ~2-3 minutes.\n')

!python data/rocstories/prepare.py --variant=all --seed=42

print('\n--- Checking output files ---')
for variant_dir in ['data/rocstories', 'data/rocstories_reverse', 'data/rocstories_struct']:
    for f in ['train.bin', 'val.bin', 'test.bin', 'meta.pkl']:
        path = os.path.join(variant_dir, f)
        if os.path.exists(path):
            size = os.path.getsize(path)
            print(f'  ✓  {path}  ({size/1e6:.1f} MB)')
        else:
            print(f'  ✗  MISSING: {path}')

## Step 4: Smoke Test (50 iterations each)

**Run this before full training.** Verifies the full pipeline works end-to-end.  
~30 seconds per model. If any cell fails, fix the issue before proceeding.

✅ Expected: loss printed every 5 steps, eval at step 25, no errors.

In [ ]:
# ── Smoke Test: M1 Forward ────────────────────────────────────────────────
import os
os.chdir('/content/demo')
print('SMOKE TEST: M1 Forward (50 iterations)...')
!python train.py config/test_m1_forward.py
print('\n✓ M1 smoke test passed!')

In [ ]:
# ── Smoke Test: M2 Reverse ────────────────────────────────────────────────
import os
os.chdir('/content/demo')
print('SMOKE TEST: M2 Reverse (50 iterations)...')
!python train.py config/test_m2_reverse.py
print('\n✓ M2 smoke test passed!')

In [ ]:
# ── Smoke Test: M3 Structured ─────────────────────────────────────────────
import os
os.chdir('/content/demo')
print('SMOKE TEST: M3 Structured (50 iterations)...')
!python train.py config/test_m3_structured.py
print('\n✓ M3 smoke test passed!')
print('\n' + '='*60)
print('All smoke tests passed! Ready for full training.')
print('='*60)

## Step 5: Full Training (5000 iterations each)

⏱ **Time estimate on free Colab T4:** ~30-60 minutes per model  
⏱ **Time estimate on Colab A100 (Pro):** ~5-10 minutes per model

Checkpoints auto-save to Google Drive when validation loss improves.

**Run M1, M2, M3 sequentially** (one cell at a time to avoid memory issues).

In [ ]:
# ── Full Training: M1 Forward Baseline ────────────────────────────────────
import os, time
os.chdir('/content/demo')

print('Starting M1 Forward training (5000 iterations)...')
print('Checkpoints saved to Drive: out-m1-forward/ckpt.pt')
print('-' * 60)

t_start = time.time()
!python train.py config/train_m1_forward.py
t_end = time.time()

mins = (t_end - t_start) / 60
print(f'\n✓ M1 training complete! Time: {mins:.1f} minutes')

# Check checkpoint was saved
ckpt = 'out-m1-forward/ckpt.pt'
if os.path.exists(ckpt):
    size = os.path.getsize(ckpt) / 1e6
    print(f'✓ Checkpoint saved: {ckpt} ({size:.1f} MB)')
else:
    print(f'WARNING: No checkpoint found at {ckpt}')

In [ ]:
# ── Full Training: M2 Reverse Causal ──────────────────────────────────────
import os, time
os.chdir('/content/demo')

print('Starting M2 Reverse training (5000 iterations)...')
print('Checkpoints saved to Drive: out-m2-reverse/ckpt.pt')
print('-' * 60)

t_start = time.time()
!python train.py config/train_m2_reverse.py
t_end = time.time()

mins = (t_end - t_start) / 60
print(f'\n✓ M2 training complete! Time: {mins:.1f} minutes')

ckpt = 'out-m2-reverse/ckpt.pt'
if os.path.exists(ckpt):
    size = os.path.getsize(ckpt) / 1e6
    print(f'✓ Checkpoint saved: {ckpt} ({size:.1f} MB)')
else:
    print(f'WARNING: No checkpoint found at {ckpt}')

In [ ]:
# ── Full Training: M3 Structured Tokens ───────────────────────────────────
import os, time
os.chdir('/content/demo')

print('Starting M3 Structured training (5000 iterations)...')
print('This is the COMPETITION SUBMISSION model.')
print('Checkpoints saved to Drive: out-m3-structured/ckpt.pt')
print('-' * 60)

t_start = time.time()
!python train.py config/train_m3_structured.py
t_end = time.time()

mins = (t_end - t_start) / 60
print(f'\n✓ M3 training complete! Time: {mins:.1f} minutes')

ckpt = 'out-m3-structured/ckpt.pt'
if os.path.exists(ckpt):
    size = os.path.getsize(ckpt) / 1e6
    print(f'✓ Checkpoint saved: {ckpt} ({size:.1f} MB)')
else:
    print(f'WARNING: No checkpoint found at {ckpt}')

## Step 6: Evaluation — Perplexity on Test Set

Compute test perplexity (PPL = exp(loss)) for all 3 models.

**Lower PPL = better.** Expected range: PPL 30-100 for a well-trained small model.

In [ ]:
import os, math, pickle, torch
import numpy as np
from contextlib import nullcontext

os.chdir('/content/demo')
sys_path = os.getcwd()
import sys
if sys_path not in sys.path:
    sys.path.insert(0, sys_path)

from model import GPT, GPTConfig

def compute_test_ppl(out_dir, dataset):
    """Load a checkpoint and compute perplexity on test.bin"""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
    ptdtype = {'bfloat16': torch.bfloat16, 'float16': torch.float16, 'float32': torch.float32}[dtype]
    ctx = torch.amp.autocast(device_type='cuda', dtype=ptdtype) if device == 'cuda' else nullcontext()

    ckpt_path = os.path.join(out_dir, 'ckpt.pt')
    if not os.path.exists(ckpt_path):
        return None, None, 'checkpoint not found'

    checkpoint = torch.load(ckpt_path, map_location=device)
    best_val_loss = checkpoint.get('best_val_loss', float('inf'))

    gptconf = GPTConfig(**checkpoint['model_args'])
    model = GPT(gptconf)
    state_dict = checkpoint['model']
    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
    model.load_state_dict(state_dict)
    model.eval().to(device)

    # Load test data
    test_bin = os.path.join('data', dataset, 'test.bin')
    if not os.path.exists(test_bin):
        return None, best_val_loss, f'test.bin not found at {test_bin}'

    data = np.memmap(test_bin, dtype=np.uint16, mode='r')
    block_size = model.config.block_size
    batch_size = 32

    total_loss = 0.0
    total_batches = 0
    n_eval = min(200, (len(data) - block_size) // batch_size)

    with torch.no_grad():
        with ctx:
            for _ in range(n_eval):
                ix = torch.randint(len(data) - block_size, (batch_size,))
                x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
                y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
                x, y = x.to(device), y.to(device)
                _, loss = model(x, y)
                total_loss += loss.item()
                total_batches += 1

    avg_loss = total_loss / total_batches
    ppl = math.exp(avg_loss)
    del model
    torch.cuda.empty_cache()
    return ppl, math.exp(best_val_loss), 'ok'


# ── Run evaluation ─────────────────────────────────────────────────────────
models = [
    ('M1 Forward',    'out-m1-forward',    'rocstories'),
    ('M2 Reverse',    'out-m2-reverse',    'rocstories_reverse'),
    ('M3 Structured', 'out-m3-structured', 'rocstories_struct'),
]

results = {}
print(f'{'Model':<20} {'Val PPL':>10} {'Test PPL':>10} {'Status'}')
print('-' * 55)

for name, out_dir, dataset in models:
    test_ppl, val_ppl, status = compute_test_ppl(out_dir, dataset)
    results[name] = {'test_ppl': test_ppl, 'val_ppl': val_ppl}
    val_str  = f'{val_ppl:.2f}'  if val_ppl  is not None else 'N/A'
    test_str = f'{test_ppl:.2f}' if test_ppl is not None else 'N/A'
    print(f'{name:<20} {val_str:>10} {test_str:>10}  {status}')

print()
print('PPL interpretation:')
print('  < 50   = excellent')
print('  50-100 = good')
print('  > 200  = undertrained or something is wrong')

## Step 7: Generate Story Samples

Generate and save samples from all 3 models. Also shows M3's guided story completion.

In [ ]:
# ── Generate samples: M1 Forward ──────────────────────────────────────────
import os
os.chdir('/content/demo')
os.makedirs('results/samples', exist_ok=True)

print('Generating 5 stories from M1 Forward model...')
print('=' * 60)
!python sample.py \
    --out_dir=out-m1-forward \
    --num_samples=5 \
    --max_new_tokens=150 \
    --temperature=0.8 \
    --top_k=100 \
    --save_to=results/samples/m1_samples.txt
print('\nSaved to results/samples/m1_samples.txt')

In [ ]:
# ── Generate samples: M2 Reverse ──────────────────────────────────────────
import os
os.chdir('/content/demo')

print('Generating 5 stories from M2 Reverse model...')
print('NOTE: Output reads S5→S1 (resolution first, setup last) — this is expected.')
print('=' * 60)
!python sample.py \
    --out_dir=out-m2-reverse \
    --num_samples=5 \
    --max_new_tokens=150 \
    --temperature=0.8 \
    --top_k=100 \
    --save_to=results/samples/m2_samples.txt
print('\nSaved to results/samples/m2_samples.txt')

In [ ]:
# ── Generate samples: M3 Structured (free generation) ────────────────────
import os
os.chdir('/content/demo')

print('Generating 5 stories from M3 Structured model (free generation)...')
print('=' * 60)
!python sample.py \
    --out_dir=out-m3-structured \
    --start='<S1>' \
    --num_samples=5 \
    --max_new_tokens=150 \
    --temperature=0.8 \
    --top_k=100 \
    --save_to=results/samples/m3_samples.txt
print('\nSaved to results/samples/m3_samples.txt')

In [ ]:
# ── M3 Guided Story Completion (BONUS capability) ─────────────────────────
import os
os.chdir('/content/demo')

print('M3 Guided Story Completion — give S1, model generates S2-S5')
print('This is only possible with M3 because of the <S1>...<S5> tokens.')
print('=' * 60)

prompts = [
    '<S1> Mary wanted to do something special for her mom. <S2>',
    '<S1> John had always been afraid of dogs. <S2>',
    '<S1> The old car broke down on the highway. <S2>',
]

for i, prompt in enumerate(prompts, 1):
    print(f'\n--- Prompt {i} ---')
    print(f'Given: {prompt}')
    print('Generated:')
    !python sample.py \
        --out_dir=out-m3-structured \
        --start="{prompt}" \
        --num_samples=2 \
        --max_new_tokens=120 \
        --temperature=0.8 \
        --top_k=100
    save_path = f'results/samples/m3_completion_{i}.txt'
    !python sample.py \
        --out_dir=out-m3-structured \
        --start="{prompt}" \
        --num_samples=3 \
        --max_new_tokens=120 \
        --temperature=0.8 \
        --top_k=100 \
        --save_to={save_path}

## Step 8: Plot Results

Extract training metrics from checkpoints and produce the loss curve plot for the report.

In [ ]:
import os, torch, math
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

os.chdir('/content/demo')

# ── Load checkpoint info ───────────────────────────────────────────────────
def load_checkpoint_info(out_dir):
    ckpt_path = os.path.join(out_dir, 'ckpt.pt')
    if not os.path.exists(ckpt_path):
        return None
    ckpt = torch.load(ckpt_path, map_location='cpu')
    return {
        'iter_num': ckpt.get('iter_num', 0),
        'best_val_loss': ckpt.get('best_val_loss', float('inf')),
        'best_val_ppl': math.exp(ckpt.get('best_val_loss', float('inf'))),
        'config': ckpt.get('config', {}),
    }

checkpoints = {
    'M1 Forward':    load_checkpoint_info('out-m1-forward'),
    'M2 Reverse':    load_checkpoint_info('out-m2-reverse'),
    'M3 Structured': load_checkpoint_info('out-m3-structured'),
}

print('Checkpoint summary:')
print(f'{'Model':<20} {'Best Val PPL':>14} {'Final Iter':>12}')
print('-' * 48)
for name, info in checkpoints.items():
    if info:
        print(f'{name:<20} {info["best_val_ppl"]:>14.2f} {info["iter_num"]:>12}')
    else:
        print(f'{name:<20} {"not trained":>14} {"N/A":>12}')

# ── Bar chart comparison ────────────────────────────────────────────────────
trained = {k: v for k, v in checkpoints.items() if v is not None}
if trained:
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    names = list(trained.keys())
    ppl_vals = [trained[n]['best_val_ppl'] for n in names]
    colors = ['#2196F3', '#F44336', '#4CAF50']

    bars = ax.bar(names, ppl_vals, color=colors[:len(names)], edgecolor='black', linewidth=0.7, width=0.5)
    for bar, ppl in zip(bars, ppl_vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{ppl:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

    ax.set_ylabel('Perplexity (lower = better)', fontsize=12)
    ax.set_title('Best Validation Perplexity — M1 vs M2 vs M3', fontsize=13, fontweight='bold')
    ax.set_ylim(0, max(ppl_vals) * 1.2)
    ax.grid(axis='y', alpha=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    os.makedirs('results', exist_ok=True)
    plt.tight_layout()
    plt.savefig('results/ppl_comparison.png', bbox_inches='tight')
    plt.show()
    print('Saved: results/ppl_comparison.png')
else:
    print('No checkpoints found — run training first (Step 5).')

## Step 9: Results Summary Table

Compile the final results table for the report.

In [ ]:
import os, torch, math
os.chdir('/content/demo')

# ── Compile summary ────────────────────────────────────────────────────────
model_info = [
    ('M1 Forward',    'out-m1-forward',    'rocstories',         'S1→S2→S3→S4→S5'),
    ('M2 Reverse',    'out-m2-reverse',    'rocstories_reverse', 'S5→S4→S3→S2→S1'),
    ('M3 Structured', 'out-m3-structured', 'rocstories_struct',  '<Si> tags'),
]

print('=' * 80)
print('FINAL RESULTS TABLE')
print('=' * 80)
print(f'{'Model':<20} {'Format':<20} {'Best Val PPL':>12} {'Final Iter':>11}')
print('-' * 70)

for name, out_dir, dataset, fmt in model_info:
    ckpt_path = os.path.join(out_dir, 'ckpt.pt')
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location='cpu')
        val_ppl = math.exp(ckpt.get('best_val_loss', float('inf')))
        iter_num = ckpt.get('iter_num', 0)
        print(f'{name:<20} {fmt:<20} {val_ppl:>12.2f} {iter_num:>11}')
    else:
        print(f'{name:<20} {fmt:<20} {"not trained":>12} {"":>11}')

print('=' * 80)
print()
print('Hypotheses:')
print('  H1: M2 (Reverse) PPL > M1 (Forward) PPL  → confirms causal asymmetry')
print('  H2: M3 (Structured) PPL ≤ M1 (Forward) PPL  → structure helps')

# Save markdown table
with open('results/perplexity_table.md', 'w') as f:
    f.write('# Perplexity Results\n\n')
    f.write('| Model | Format | Best Val PPL | Final Iter |\n')
    f.write('|-------|--------|-------------|-----------|\n')
    for name, out_dir, dataset, fmt in model_info:
        ckpt_path = os.path.join(out_dir, 'ckpt.pt')
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location='cpu')
            val_ppl = math.exp(ckpt.get('best_val_loss', float('inf')))
            iter_num = ckpt.get('iter_num', 0)
            f.write(f'| {name} | {fmt} | {val_ppl:.2f} | {iter_num} |\n')
        else:
            f.write(f'| {name} | {fmt} | N/A | N/A |\n')

print('\nSaved: results/perplexity_table.md')

## Step 10: Upload M3 to HuggingFace (Task 3 Submission)

Upload your best model (M3 structured) for the competition submission.

In [ ]:
# ── Create sampling_params.json for competition submission ────────────────
import json, os
os.chdir('/content/demo')

sampling_params = {
    'temperature': 0.8,
    'top_k': 100,
    'max_new_tokens': 200,
    'start': '<S1>'
}

with open('out-m3-structured/sampling_params.json', 'w') as f:
    json.dump(sampling_params, f, indent=2)

print('Created: out-m3-structured/sampling_params.json')
print(json.dumps(sampling_params, indent=2))

# Also copy model.py to checkpoint dir (required for submission)
import shutil
shutil.copy('model.py', 'out-m3-structured/model.py')
print('Copied: model.py → out-m3-structured/model.py')

print('\nSubmission directory contents:')
for f in os.listdir('out-m3-structured'):
    path = os.path.join('out-m3-structured', f)
    size = os.path.getsize(path)
    print(f'  {f:<30} ({size/1e6:.2f} MB)')

In [ ]:
# ── Upload to HuggingFace ─────────────────────────────────────────────────
# Replace with your HuggingFace username and token
HF_USERNAME = 'YOUR_HF_USERNAME'
HF_TOKEN    = 'hf_YOUR_TOKEN_HERE'
REPO_NAME   = 'nanoGPT_hw'

!pip install -q huggingface_hub

from huggingface_hub import HfApi
api = HfApi()

# Create repo if it doesn't exist
repo_id = f'{HF_USERNAME}/{REPO_NAME}'
try:
    api.create_repo(repo_id=repo_id, token=HF_TOKEN, exist_ok=True)
    print(f'Repo ready: https://huggingface.co/{repo_id}')
except Exception as e:
    print(f'Repo creation error (may already exist): {e}')

# Upload the required files
upload_files = [
    ('out-m3-structured/ckpt.pt', 'ckpt.pt'),
    ('out-m3-structured/model.py', 'model.py'),
    ('out-m3-structured/sampling_params.json', 'sampling_params.json'),
]

import os
os.chdir('/content/demo')

for local_path, repo_path in upload_files:
    if os.path.exists(local_path):
        print(f'Uploading {local_path} → {repo_path} ...')
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_path,
            repo_id=repo_id,
            token=HF_TOKEN,
        )
        print(f'  ✓ Uploaded {repo_path}')
    else:
        print(f'  ✗ NOT FOUND: {local_path}')

print(f'\nSubmission URL: https://huggingface.co/{repo_id}')
print('Submit this URL + your HF token to Canvas.')

---

## Quick Reference: All Commands

```bash
# ─── Data preparation ───────────────────────────────────────────────────────
# Prepare one variant:
python data/rocstories/prepare.py --variant=forward
python data/rocstories/prepare.py --variant=reverse
python data/rocstories/prepare.py --variant=structured

# Prepare all 3 at once:
python data/rocstories/prepare.py --variant=all

# ─── Smoke tests (50 iterations) ────────────────────────────────────────────
python train.py config/test_m1_forward.py
python train.py config/test_m2_reverse.py
python train.py config/test_m3_structured.py

# ─── Full training (5000 iterations) ────────────────────────────────────────
python train.py config/train_m1_forward.py
python train.py config/train_m2_reverse.py
python train.py config/train_m3_structured.py

# ─── Generate samples ───────────────────────────────────────────────────────
# M1 / M2 — free generation:
python sample.py --out_dir=out-m1-forward --num_samples=5 --max_new_tokens=150
python sample.py --out_dir=out-m2-reverse --num_samples=5 --max_new_tokens=150

# M3 — guided story completion:
python sample.py --out_dir=out-m3-structured --start='<S1> Mary went to school. <S2>' \
    --num_samples=3 --max_new_tokens=150 --temperature=0.8 --top_k=100

# Save samples to file:
python sample.py --out_dir=out-m1-forward --save_to=results/samples/m1_samples.txt

# ─── Evaluation (perplexity on eval stories) ─────────────────────────────────
python eval.py --init_from=resume --out_dir=out-m1-forward \
    --input_file=data/rocstories/eval_stories.txt
```

---

## Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Reduce `batch_size=32` in config, or restart runtime |
| `No module named tiktoken` | Run `!pip install tiktoken` |
| `CSV not found` | Run Step 3 again |
| `ckpt.pt not found` | Training didn't save — check `always_save_checkpoint=True` in config |
| Compilation error (`torch.compile`) | Add `--compile=False` to your train command |
| Session crashed, no checkpoint | Checkpoints are on Drive — relink symlinks in Step 2 and resume |